# Daily reconciliation — <strategy name>

Fill one copy of this notebook per 30-day live paper run.

**The rule:** every row with `status != OK` must have a written `explanation`.
A blank explanation is an unexplained gap, and unexplained gaps are the thing
this whole exercise exists to eliminate.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 140)

rec = pd.read_csv('live/reconciliation.csv', parse_dates=['date'])
log = pd.read_csv('live/daily-log.csv', parse_dates=['date'])
print(rec.shape, rec['date'].min(), rec['date'].max())

## 1. Daily status counts

In [ ]:
counts = rec.pivot_table(index='date', columns='status', values='symbol',
                         aggfunc='count', fill_value=0)
counts.plot(kind='bar', stacked=True, figsize=(12, 3), title='status per day');
counts.tail()

## 2. Unexplained gap  — this must be 0.000

The single number the grader checks.

In [ ]:
gaps = rec[(rec['status'] != 'OK')]
unexplained = gaps[gaps['explanation'].isna() | (gaps['explanation'].str.strip() == '')]
print(f"rows needing explanation : {len(gaps)}")
print(f"unexplained              : {len(unexplained)}")
print(f"cumulative |diff_weight| unexplained : {unexplained['diff_weight'].abs().sum():.4f}")
assert len(unexplained) == 0, 'explain every gap before you finish'
unexplained

## 3. Slippage: realized vs assumed

Your backtests assumed 5 bps. Compare.

In [ ]:
ASSUMED_BPS = 5.0
s = rec['slippage_bps'].dropna()
print(f"n fills {len(s)}   mean {s.mean():.2f} bps   median {s.median():.2f}   p95 {s.quantile(.95):.2f}")
print(f"backtest assumed {ASSUMED_BPS:.1f} bps -> realized is {s.mean()/ASSUMED_BPS:.2f}x")
s.hist(bins=30, figsize=(9,3));
plt.axvline(ASSUMED_BPS, color='r', ls='--', label='assumed'); plt.legend();

## 4. Intended vs actual equity

In [ ]:
ax = log.set_index('date')[['intended_equity','actual_equity']].plot(figsize=(11,4))
gap = (log['actual_equity'].iloc[-1] / log['intended_equity'].iloc[-1] - 1)
print(f"execution gap over the run: {gap*100:+.2f}%")
ax.set_title('intended vs actual equity');

## 5. Fill rate and turnover

In [ ]:
log['fill_rate'] = log['orders_filled'] / log['orders_submitted'].replace(0, np.nan)
print(log[['date','orders_submitted','orders_filled','fill_rate','turnover']].tail(10).to_string(index=False))
print()
print(f"mean live turnover   : {log['turnover'].mean():.4f}")
print(f"backtest turnover    : <fill in from your tear sheet>")
print('If live > 1.5x backtest, your live signal differs from your backtested one.')

## 6. Incidents

Every row here also gets an entry in `live/incidents.md` with a root cause.

In [ ]:
incidents = rec[rec['status'].isin(['NEW','MISSING','SIDE_DIFF'])]
print(incidents.groupby('status').size())
incidents.head(20)

## 7. The write-up

Answer in `live/live-log.md`, in prose:
1. How many days had zero gaps? What caused the rest?
2. Did realized slippage match your backtest assumption? If not, restate the
   affected verdicts' cost assumptions.
3. Did anything happen in live trading that your backtester does not model?
   (This list is the most valuable output of Week 12.)